# Boosting vs Bagging vs Stacking

We will use the MNIST dataset for this example, train all the three approaches and compare the accuracy score on the test set. 

we will also highlight the reasoning - why anyone model works best on this datset.

In [3]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
# Features (images)
X = mnist.data  

# Labels (digits)
y = mnist.target

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 10000)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 10000)


## Training a Bagging algorithm (Random Forest)

First: trying to train a random forecast with max_depth = 3, so dont want to overfit the data. However, this looks very low as we have a tough problem and a lot of data to train on.  
Bagging reduce the variance of the base model and prevents overfitting. However, nothing can be said about bias, as bias will depent on the base model

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
clf = RandomForestClassifier(max_depth=3, n_estimators = 200, random_state=0)
clf.fit(X_train, y_train)
y_pred_val   = clf.predict(X_val)
print("Accuracy of Random Forest :", accuracy_score(y_val, y_pred_val))

Accuracy of Random Forest : 0.7466


The above random forest is providing us a poor result of accuracy 0.75. we can use k-fold cross validation to find the best hyperparamers

In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
params = {'max_depth': [4, 6, 9, 10, 20, 30], 'n_estimators': [100, 150, 200]}
clf = RandomForestClassifier(random_state=0)
grid_search= GridSearchCV(clf, params, verbose=1, cv=5)
grid_search.fit(X_train, y_train)
best_clf = grid_search.best_estimator_
best_clf.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


KeyboardInterrupt: 

In [ ]:
print("Accuracy of the best Random forest is :", best_clf.score(X_val, y_val))

Accuracy of the best Random forest is : 0.9711


## Now Training the Gradient Bosting model 

Gradient boosting in its raw version will take a lot of time to learn. so here, we are training an approximate metthod which is faster using HistGradientBoostingClassifier, this class of sklearn ust approximate the split finding step and make it faster by binning the search space. \
Boosting reduces both bias and variance

In [14]:
from xgboost import XGBClassifier

# create model instance
bst = XGBClassifier(n_estimators=50, max_depth=30, learning_rate=0.3, objective='multi:softprob')
# fit model
y_train = y_train.astype(int)
y_val = y_val.astype(int)

bst.fit(X_train, y_train)
# make predictions
preds = bst.predict(X_val)
print("Accuracy of Random Forest :", accuracy_score(y_val, preds))


Accuracy of Random Forest : 0.9722


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(learning_rate=0.1,
early_stopping = True,
    max_depth=30, random_state=0).fit(X_train, y_train)
print("Accuracy of Gradient Boosting :" , clf.score(X_val, y_val))


Accuracy of Gradient Boosting : 0.9764


We Can notice here that, Gradient boosting is a little accurate than the best random forest. 
Note - we are using an approximated gradient boosting algorithm which is not even optimized and still works better than random forecast (bagging)

# Now training a stacking classifier